# 🚀 Prometheus Quant Engine: Quickstart & Double-Spend Protection
Welcome to the Prometheus Quant Engine API.

This notebook demonstrates how to price standard European options using our C++ HPC cluster directly from Python.

### 🔑 Step 1: Claim Your Free API Key
To run this notebook, you need a live API key. Head over to **[prometheusquantengine.com](https://prometheusquantengine.com)** and register. 
* Every new developer account is instantly provisioned with **50 Compute Credits** (enough to simulate 12.5 Billion paths for free). No credit card required.
Once you have your key, paste it in the `API_KEY` variable below.

### 💰 Compute Ledger Mechanics (Absolute Transparency)
Prometheus does not charge per API request; we charge strictly for raw mathematical computation.
The formula is: **Total Steps = N (Simulations) × M (Time Steps)**.
* **Cost Rate:** 250,000,000 stochastic steps = 1.0000 Compute Credit.
* European Options are path-independent, meaning `M = 1`. 

Let's price a European Call with 10,000,000 paths.
* **Cost Calculation:** (10,000,000 * 1) / 250,000,000 = **0.04 Credits**.

In [ ]:
import requests
import uuid
import time

API_KEY = "pmt_live_..." # 👈 PASTE YOUR API KEY HERE
BASE_URL = "https://api.prometheusquantengine.com/api/v1/simulations"

# Generate a unique Idempotency Key to protect against network timeouts
idem_key = str(uuid.uuid4())

headers = {
    "X-API-Key": API_KEY,
    "Idempotency-Key": idem_key,
    "Content-Type": "application/json"
}

payload = {
    "simulation_type": "European",
    "s_0": 100.0,
    "strike": 100.0,
    "volatility": 0.20,
    "time_to_maturity": 1.0,
    "risk_free_rate": 0.05,
    "option_type": "Call",
    "n_simulations": 10000000, # 10 Million Paths
    "label": "European_Call_Colab"
}

print(f"Submitting 10 Million Paths to the C++ Engine...")
start_time = time.time()
response = requests.post(BASE_URL, json=payload, headers=headers)
print(f"Completed in {time.time() - start_time:.4f} seconds!\n")

data = response.json()
print(f"Fair Value: {data.get('fair_value')}")
print(f"Delta (Δ): {data.get('delta')}")
print(f"Credits Deducted: {data.get('credits_cost')} Cr")

### 🛡️ Testing Idempotency (Zero-Cost Retries)
What happens if your WiFi drops and you run the exact same cell again? If we pass the same `Idempotency-Key` with the same payload, Prometheus bypasses the engine and returns the cached result for **0.00 Credits**. Let's prove it:

In [ ]:
# Re-sending the exact same request with the exact same Idempotency Key
print("Re-submitting payload...")
retry_response = requests.post(BASE_URL, json=payload, headers=headers)
retry_data = retry_response.json()

print(f"Fair Value: {retry_data.get('fair_value')}")
print(f"Credits Deducted: {retry_data.get('credits_cost')} Cr (Notice it's 0.0!)")